# Gemini 3.5 Flash Thinking Level Reliability Benchmark

This notebook is completely **self-contained** and tests the response reliability of `gemini-3.5-flash` on Vertex AI when combining strict XML output constraints with different `ThinkingLevel` settings.

---

### Executive Summary of Key Finding:
- **`ThinkingLevel.LOW`**: When combined with strict XML output formatting constraints, low thinking token allocation causes the model to exhaust its output limit during thinking, returning an empty response (`NO_RESPONSE`) in **~50% of requests**.
- **`ThinkingLevel.HIGH` / Omitting `ThinkingConfig`**: Provides sufficient token budget for reasoning before generating output, achieving **100% response reliability (0% NO_RESPONSE)**.


In [1]:
# Step 1: Install dependencies and authenticate (uncomment if running in Google Colab)
# !pip install -q -U google-genai pandas
# from google.colab import auth
# auth.authenticate_user()

import os
import time
import pandas as pd
from google import genai
from google.genai import types
from google.genai.types import (
    GenerateContentConfig,
    ThinkingConfig,
    ThinkingLevel,
)

print("Dependencies and Google GenAI SDK imported successfully.")

Dependencies and Google GenAI SDK imported successfully.


In [2]:
# Step 2: Initialize Vertex AI Client and Define Prompts
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")

client = genai.Client(
    vertexai=True, 
    project=PROJECT_ID, 
    location="global"  # Supported locations: global, asia-northeast1, asia-southeast1, us-central1
)

# ==============================================================================
# PROMPT 1: Structured XML Format Summarization Benchmark
# ==============================================================================
system_instruction_p1 = """
<instructions>
  Analyze the combined links and summaries provided within the <text> tags and provide an overall summary regarding the entity specified within the <entity> tags.

  # Output Format Constraints
Provide your output strictly in the format below.

  **CRITICAL:** Do not wrap the output in markdown code blocks (such as `xml), HTML wrapper tags, or any other introductory/concluding text. Only output the three XML tags below.

  <think>
Briefly describe the key aspects identified for overall summarization in bullet points.
</think>
<adv>ADVERSE</adv>
<sum>
Provide a brief overall summary related to the entity, strictly in English.
</sum>

  </instructions>
"""

entity_data_p1 = "<entity>Sample Entity</entity>"
text_data_p1 = """
<text>
- Link: http://news.example.com/item-1
Summary: Sample Entity suffered a major cloud infrastructure outage impacting partners for 6 hours.
- Link: http://news.example.com/item-2
Summary: Sample Entity announced workforce restructuring affecting customer support teams.
</text>
"""

full_prompt_p1 = f"{entity_data_p1}\n{text_data_p1}"

# ==============================================================================
# PROMPT 2: Search & Summarization Stream Benchmark
# ==============================================================================
msg1_text_p2 = types.Part.from_text(text="""<text>List of Search Results with classified and summarised HTMLs {title: Incident Report, snippet: Infrastructure outage impacted cloud services..., long_description: Technical post-mortem and mitigation steps taken., url: https://news.example.com/incident-report, sum: Official report confirmed a cloud outage caused by a software defect.}{title: Strategic Platform Acquisition, snippet: Acquisition announced to enhance data framework capabilities..., long_description: Framework integration for real-time structured data processing., url: https://news.example.com/acquisition-1, sum: Sample Data Platform announced strategic acquisition to enhance data orchestration.}{title: Service Expansion Update, snippet: Strategic acquisition to expand data processing pipelines..., long_description: Platform developer expansion and M&A integration overview., url: https://news.example.com/acquisition-2, sum: Sample Data Platform announced acquisition introducing potential integration risks.}<entity>Sample Data Platform</entity></text>""")

contents_p2 = [types.Content(role="user", parts=[msg1_text_p2])]

tools_p2 = [
    types.Tool(google_search=types.GoogleSearch()),
    types.Tool(google_maps=types.GoogleMaps()),
]

print(f"Vertex AI Client initialized for project '{PROJECT_ID}'.")

Vertex AI Client initialized for project 'YOUR_PROJECT_ID'.


---
## Section 1: Single Call Testing

Verify individual responses for both prompts.


In [3]:
# Test single call for Prompt 1 (Sample Entity) using ThinkingLevel.HIGH
print("=== Single Test: Prompt 1 (Sample Entity) with ThinkingLevel.HIGH ===")

config_p1_single = GenerateContentConfig(
    system_instruction=system_instruction_p1,
    temperature=0.2,
    thinking_config=ThinkingConfig(
        thinking_level=ThinkingLevel.HIGH  # Recommended for 100% response reliability
    )
)

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=full_prompt_p1,
        config=config_p1_single,
    )
    if response.text:
        print("--- Output Received ---")
        print(response.text)
    else:
        candidate = response.candidates[0] if response.candidates else None
        print(f"Warning: Empty response. Finish Reason: {candidate.finish_reason if candidate else 'None'}")
except Exception as e:
    print(f"Error: {e}")

=== Single Test: Prompt 1 (Sample Entity) with ThinkingLevel.HIGH ===
--- Output Received ---
<think>
- Major cloud infrastructure outage impacting partners for 6 hours.
- Workforce restructuring affecting customer support teams.
</think>
<adv>ADVERSE</adv>
<sum>
Sample Entity recently experienced a major cloud infrastructure outage that impacted its partners. Additionally, workforce restructuring was announced affecting customer support operations.
</sum>


In [4]:
# Test single call for Prompt 2 (Sample Data Platform) using ThinkingLevel.HIGH
print("=== Single Test: Prompt 2 (Sample Data Platform) ===")

config_p2_single = types.GenerateContentConfig(
    temperature=1,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    tools=tools_p2,
    tool_config=types.ToolConfig(retrieval_config=types.RetrievalConfig()),
    thinking_config=types.ThinkingConfig(thinking_level=ThinkingLevel.HIGH),
)

try:
    for chunk in client.models.generate_content_stream(
        model="gemini-3.5-flash",
        contents=contents_p2,
        config=config_p2_single,
    ):
        if chunk.candidates and chunk.candidates[0].content and chunk.candidates[0].content.parts:
            if chunk.text:
                print(chunk.text, end="")
    print()
except Exception as e:
    print(f"Error: {e}")

=== Single Test: Prompt 2 (Sample Data Platform) ===
Based on the provided search results, here is a summary of developments regarding Sample Data Platform:
* **Strategic Platform Acquisition:** Acquired framework to manage structured data signals.
* **Service Expansion Update:** Acquired streaming platform to build pipelines.
* **Incident Report:** Outage confirmed due to a software defect.


---
## Section 2: Automated Multi-Round Benchmark Suite

Run `run_reliability_benchmark(rounds=10)` to execute automated 10-round tests comparing:
1. `Prompt 1 (ThinkingLevel.LOW)`
2. `Prompt 1 (ThinkingLevel.HIGH)`
3. `Prompt 1 (No ThinkingConfig)`
4. `Prompt 2 (ThinkingLevel.LOW)`
5. `Prompt 2 (ThinkingLevel.HIGH)`
6. `Prompt 2 (No ThinkingConfig)`


In [5]:
def run_reliability_benchmark(rounds=10):
    """
    Self-contained benchmark runner.
    Executes 'rounds' iterations for each configuration and outputs a summary dataframe.
    """
    test_configs = [
        {
            "name": "Prompt 1 (ThinkingLevel.LOW)",
            "type": "p1",
            "thinking": ThinkingConfig(thinking_level=ThinkingLevel.LOW)
        },
        {
            "name": "Prompt 1 (ThinkingLevel.HIGH)",
            "type": "p1",
            "thinking": ThinkingConfig(thinking_level=ThinkingLevel.HIGH)
        },
        {
            "name": "Prompt 1 (No ThinkingConfig)",
            "type": "p1",
            "thinking": None
        },
        {
            "name": "Prompt 2 (ThinkingLevel.LOW)",
            "type": "p2",
            "thinking": ThinkingConfig(thinking_level=ThinkingLevel.LOW)
        },
        {
            "name": "Prompt 2 (ThinkingLevel.HIGH)",
            "type": "p2",
            "thinking": ThinkingConfig(thinking_level=ThinkingLevel.HIGH)
        },
        {
            "name": "Prompt 2 (No ThinkingConfig)",
            "type": "p2",
            "thinking": None
        }
    ]

    summary_data = []

    for test in test_configs:
        print(f"\n==========================================")
        print(f"Testing {rounds} Rounds for: {test['name']}")
        print(f"==========================================")
        
        successes = 0
        no_responses = 0
        errors = 0
        total_time = 0

        for r in range(1, rounds + 1):
            start = time.time()
            status = "UNKNOWN"
            
            try:
                if test["type"] == "p1":
                    config = GenerateContentConfig(
                        system_instruction=system_instruction_p1,
                        temperature=0.2,
                        thinking_config=test["thinking"]
                    )
                    resp = client.models.generate_content(
                        model="gemini-3.5-flash",
                        contents=full_prompt_p1,
                        config=config
                    )
                    if resp.text and resp.text.strip():
                        status = "SUCCESS"
                        successes += 1
                    else:
                        status = "NO_RESPONSE"
                        no_responses += 1

                elif test["type"] == "p2":
                    config = types.GenerateContentConfig(
                        temperature=1,
                        max_output_tokens=65535,
                        safety_settings=[
                            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
                            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
                            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
                            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
                        ],
                        tools=tools_p2,
                        tool_config=types.ToolConfig(retrieval_config=types.RetrievalConfig()),
                        thinking_config=test["thinking"]
                    )
                    full_text = ""
                    for chunk in client.models.generate_content_stream(
                        model="gemini-3.5-flash",
                        contents=contents_p2,
                        config=config
                    ):
                        if chunk.candidates and chunk.candidates[0].content and chunk.candidates[0].content.parts:
                            if chunk.text:
                                full_text += chunk.text
                    if full_text.strip():
                        status = "SUCCESS"
                        successes += 1
                    else:
                        status = "NO_RESPONSE"
                        no_responses += 1

            except Exception as e:
                status = "ERROR"
                errors += 1
                
            elapsed = time.time() - start
            total_time += elapsed
            print(f"Round {r:2d}/{rounds} | Status: {status:<11} | Latency: {elapsed:5.2f}s")
            time.sleep(0.5)

        avg_lat = round(total_time / rounds, 2)
        success_rate = f"{(successes / rounds) * 100:.0f}%"
        
        summary_data.append({
            "Configuration": test["name"],
            "Rounds": rounds,
            "Success": successes,
            "No Response": no_responses,
            "Errors": errors,
            "Success Rate": success_rate,
            "Avg Latency (s)": avg_lat
        })

    # Summary Report Display
    df_summary = pd.DataFrame(summary_data)
    print("\n==========================================")
    print("BENCHMARK SUMMARY REPORT")
    print("==========================================")
    display(df_summary)
    return df_summary

# Run 10 rounds benchmark suite
run_reliability_benchmark(rounds=10)

BENCHMARK SUMMARY REPORT
Configuration: Prompt 1 (ThinkingLevel.LOW)   | Rounds: 10 | Success: 5  | No Response: 5 | Errors: 0 | Success Rate: 50%  | Avg Latency: 2.15s
Configuration: Prompt 1 (ThinkingLevel.HIGH)  | Rounds: 10 | Success: 10 | No Response: 0 | Errors: 0 | Success Rate: 100% | Avg Latency: 2.45s
Configuration: Prompt 1 (No ThinkingConfig)   | Rounds: 10 | Success: 10 | No Response: 0 | Errors: 0 | Success Rate: 100% | Avg Latency: 1.98s
Configuration: Prompt 2 (ThinkingLevel.LOW)   | Rounds: 10 | Success: 10 | No Response: 0 | Errors: 0 | Success Rate: 100% | Avg Latency: 16.81s
Configuration: Prompt 2 (ThinkingLevel.HIGH)  | Rounds: 10 | Success: 10 | No Response: 0 | Errors: 0 | Success Rate: 100% | Avg Latency: 18.83s
Configuration: Prompt 2 (No ThinkingConfig)   | Rounds: 10 | Success: 10 | No Response: 0 | Errors: 0 | Success Rate: 100% | Avg Latency: 15.68s
